# Phase 3 — Auto-labeling (AIS → SAR)

Generates YOLO training labels with no manual annotation. For each Sentinel-1
scene we know the acquisition second, so we know where every AIS-broadcasting
vessel stood when the radar looked.

**Kill criterion:** under ~70% of boxes on real targets means an overpass-time
or geolocation bug — stop and fix before training.

Needs no GPU. It *is* network- and disk-heavy: each AIS day is ~650 MB.


In [ ]:
# Clone (or pull) and install. Re-run this cell after every `git push`.
import os, pathlib
if pathlib.Path('smugglers').exists():
    !cd smugglers && git pull --ff-only
else:
    !git clone https://github.com/Jaswanth-K1210/smugglers.git
%cd /content/smugglers
!pip install -q -r requirements.txt
print()
!git log --oneline -1


In [ ]:
# Credentials. Only GFW needs one — SAR and AIS are open.
# Colab: use the key icon in the left sidebar, add GFW_API_TOKEN, then run this.
from google.colab import userdata
import pathlib
tok = ""
try:
    tok = userdata.get('GFW_API_TOKEN')
except Exception:
    print("No Colab secret found. Paste your token below if you want GFW layers.")
pathlib.Path('.env').write_text(
    f"GFW_API_TOKEN={tok}\nAOI_BBOX=9.5,56.5,12.5,58.5\n")
from src.config import GFW_API_TOKEN, bbox
print("GFW token set:", bool(GFW_API_TOKEN), "| AOI:", bbox())


## 1. One scene, to check the machinery

Labels one scene and prints the contrast of every box against its local background.

In [ ]:
from src import fetch_s1, autolabel
import pandas as pd

SKAGEN = (10.4, 57.5, 11.4, 58.0)     # the anchorage, water-heavy
items = fetch_s1.search('2025-06-08', '2025-06-08', box=SKAGEN)
best  = max(items, key=lambda i: fetch_s1.overlap(i, SKAGEN))
tif   = fetch_s1.fetch(best, box=SKAGEN)
print('scene time:', autolabel.scene_time(tif))


In [ ]:
# Phase 1 events first, so `sts` boxes can be formed.
from src import ais_sts
from src.filters import drop_coastal
from src.config import DATA

dates  = ['2025-06-08']
events = drop_coastal(ais_sts.sts_events(dates, box=SKAGEN))
print(len(events), 'at-sea STS events')
boxes  = autolabel.label_scene(tif, events, box=SKAGEN)
print(len(boxes), 'boxes:', boxes.cls.value_counts().to_dict())
boxes.head(10)


### Kill-criterion check

Objective, not eyeballed: how far above the local background does each box sit?

In [ ]:
import numpy as np, rasterio
with rasterio.open(tif) as src:
    img = src.read(1).astype(float)
db = []
for _, b in boxes.iterrows():
    r0,c0,r1,c1 = int(b.r0),int(b.c0),int(b.r1),int(b.c1)
    inner = img[r0:r1+1, c0:c1+1]
    outer = img[max(0,r0-40):r1+40, max(0,c0-40):c1+40]
    bg = np.median(outer[outer>0])
    db.append(20*np.log10(inner.max()/max(bg,1)))
db = np.array(db)
rate = (db >= 13).mean()
print(f"median contrast {np.median(db):.1f} dB, weakest {db.min():.1f} dB")
print(f"{rate:.0%} of boxes >= 13 dB above background ->", "PASS" if rate >= .70 else "FAIL")


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(autolabel.preview_boxes(tif, boxes))))


## 2. Batch — the dataset that actually matters

One scene gives ~20 tiles, which is far too few. This labels every scene over
the AOI in a date range.

**AIS is the cost**: one ~650 MB download per distinct date. Seven days is
~4.6 GB. Start with 2–3 days to confirm it runs, then widen.

In [ ]:
from src.autolabel import label_many
summary = label_many('2025-06-06', '2025-06-08', box=SKAGEN, max_scenes=8)
summary


In [ ]:
!ls data/tiles/images | wc -l
!ls data/tiles/labels | head -3


## 3. Copy the tiles somewhere durable

Colab wipes `/content` when the runtime ends. Tiles are the expensive artifact —
keep them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/darksts && cp -r data/tiles /content/drive/MyDrive/darksts/
!du -sh /content/drive/MyDrive/darksts/tiles


---
**Report back:** the PASS/FAIL line, the tile count, and the chip sheet. If the
kill criterion fails, paste the whole cell output — the fix is upstream in
overpass timing or geolocation, not in the threshold.